## 3- Explicabilidad (XAI)

En este apartado abordamos la problemática de la **explicabilidad** (*Explainable AI, XAI*) aplicada al modelo de predicción de readmisión hospitalaria en pacientes diabéticos.

La explicabilidad es especialmente relevante en este contexto por dos motivos:
- **Ético y legal**: el modelo toma decisiones que afectan directamente a pacientes. El personal médico necesita entender *por qué* el modelo predice un alto riesgo de readmisión para poder actuar en consecuencia. Además, el RGPD (Art. 22) recoge el derecho a la explicación en decisiones automatizadas.
- **Clínico**: las predicciones solo son útiles si se apoyan en variables médicamente razonables. Si el modelo se basa en proxies espurios (ej. número de encuentros previos como proxy de raza), su despliegue puede ser perjudicial.

Se aplican dos técnicas complementarias:
1. **SHAP** (*SHapley Additive exPlanations*, Lundberg & Lee, 2017): explicaciones globales y locales basadas en teoría de juegos cooperativos.
2. **LIME** (*Local Interpretable Model-agnostic Explanations*, Ribeiro et al., 2016): explicaciones locales mediante un modelo lineal aproximado en el entorno de cada instancia.

El modelo base utilizado es **XGBoost**, que es el modelo con mejor rendimiento obtenido en el pipeline principal del proyecto.

### 3.1 Imports y carga de datos

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

import shap
from lime.lime_tabular import LimeTabularExplainer

In [ ]:
# Semilla para reproducibilidad
SEED = 22
np.random.seed(SEED)

In [2]:
# Carga del dataset preprocesado
df = pd.read_csv('../data/diabetes_preprocesado.csv', sep=',', quotechar='"')

# Las columnas 'encounter_id' y 'patient_nbr' son identificadores, no features clínicas
# Las eliminamos para evitar data leakage
cols_to_drop = ['encounter_id', 'patient_nbr']
df = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

y = df['readmitted']
X = df.drop(columns=['readmitted'])

feature_names = list(X.columns)

print(f'Dataset: {X.shape[0]} instancias, {X.shape[1]} features')
print(f'\nDistribución de clases:')
print(y.value_counts().rename({0: 'No readmitido', 1: '>30 días', 2: '<30 días'}))

FileNotFoundError: [Errno 2] No such file or directory: '../data/diabetes_preprocesado.csv'

In [ ]:
# Split estratificado (mismo que en el resto del proyecto)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print(f'Entrenamiento: {X_train.shape[0]} instancias')
print(f'Test:          {X_test.shape[0]} instancias')

### 3.2 Entrenamiento del modelo (XGBoost)

Entrenamos un clasificador **XGBoost** que actúa como modelo a explicar. Se usa `scale_pos_weight` para compensar parcialmente el desbalanceo (alternativa a SMOTE cuando no se quiere modificar los datos originales para el análisis de explicabilidad).

In [ ]:
# XGBoost con hiperparámetros razonables para el problema
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=SEED,
    n_jobs=-1
)

xgb_model.fit(X_train, y_train)

y_pred = xgb_model.predict(X_test)
print('Rendimiento del modelo XGBoost (conjunto de test):')
print(classification_report(
    y_test, y_pred,
    target_names=['No readmitido (0)', '>30 días (1)', '<30 días (2)']
))
print(f'F1-macro: {f1_score(y_test, y_pred, average="macro"):.4f}')

---
### 3.3 Explicabilidad Global con SHAP

SHAP asigna a cada feature una contribución (*Shapley value*) basada en su impacto marginal promedio a lo largo de todas las posibles combinaciones de features. Para modelos de árbol como XGBoost, `TreeExplainer` calcula los SHAP values de forma exacta y eficiente.

Dado que el problema es **multiclase** (3 clases), SHAP genera un valor por clase y feature. Analizaremos principalmente la clase de interés clínico: **readmisión temprana (<30 días, clase 2)**.

In [ ]:
# TreeExplainer es el más eficiente para modelos basados en árboles (XGBoost, RF...)
explainer = shap.TreeExplainer(xgb_model)

# Calculamos SHAP values sobre el conjunto de test completo
# shap_values tiene shape (n_clases, n_samples, n_features) para multiclase
shap_values = explainer.shap_values(X_test)

# Para modelos multiclase en XGBoost, shap_values es una lista de arrays (uno por clase)
# Clase 0: No readmitido | Clase 1: >30 días | Clase 2: <30 días (readmisión temprana)
print(f'Tipo de shap_values: {type(shap_values)}')
if isinstance(shap_values, list):
    print(f'Número de clases: {len(shap_values)}')
    print(f'Shape por clase: {shap_values[0].shape}')
else:
    print(f'Shape: {shap_values.shape}')

In [ ]:
# --- BEESWARM PLOT (Summary Plot): importancia global + dirección del efecto ---
# Mostramos las top 15 features más importantes para predecir readmisión <30 días (clase 2)

# Índice de la clase de interés clínico: readmisión temprana
CLASS_IDX = 2
CLASS_NAME = 'Readmisión <30 días (clase 2)'

if isinstance(shap_values, list):
    sv_class = shap_values[CLASS_IDX]
else:
    sv_class = shap_values[:, :, CLASS_IDX]

plt.figure(figsize=(10, 8))
shap.summary_plot(
    sv_class,
    X_test,
    feature_names=feature_names,
    max_display=15,
    show=False,
    plot_type='dot'   # beeswarm
)
plt.title(f'SHAP Beeswarm Plot — {CLASS_NAME}', fontsize=13, pad=15)
plt.tight_layout()
plt.savefig('../visualizacion_problematicas/shap_beeswarm_clase2.png', dpi=150, bbox_inches='tight')
plt.show()
print('Interpretación: cada punto es una instancia del test. El eje X muestra el SHAP value '
      '(impacto en la predicción). El color indica el valor de la feature (rojo=alto, azul=bajo).')

In [ ]:
# --- BAR PLOT: importancia media |SHAP| por clase ---
# Permite comparar qué features son globalmente más importantes para el modelo

class_names = ['No readmitido (0)', '>30 días (1)', '<30 días (2)']

if isinstance(shap_values, list):
    n_classes = len(shap_values)
    mean_abs_shap = np.array([np.abs(shap_values[c]).mean(axis=0) for c in range(n_classes)])
else:
    n_classes = shap_values.shape[2]
    mean_abs_shap = np.array([np.abs(shap_values[:, :, c]).mean(axis=0) for c in range(n_classes)])

# Ordenar por importancia media global (promedio entre clases)
global_importance = mean_abs_shap.mean(axis=0)
top_idx = np.argsort(global_importance)[::-1][:15]
top_features = [feature_names[i] for i in top_idx]

colors = ['#4C8BE0', '#E07C4C', '#4CBE8C']
x = np.arange(len(top_features))
width = 0.25

fig, ax = plt.subplots(figsize=(14, 6))
for i, (c, name) in enumerate(zip(range(n_classes), class_names)):
    ax.bar(x + i * width, mean_abs_shap[c][top_idx], width, label=name, color=colors[i], alpha=0.85)

ax.set_xlabel('Feature', fontsize=11)
ax.set_ylabel('Importancia media |SHAP|', fontsize=11)
ax.set_title('Importancia Global SHAP por Clase (Top 15 features)', fontsize=13)
ax.set_xticks(x + width)
ax.set_xticklabels(top_features, rotation=45, ha='right', fontsize=9)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../visualizacion_problematicas/shap_bar_por_clase.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- DEPENDENCE PLOT: efecto de las 2 features más importantes para clase 2 ---
# Muestra cómo varía el SHAP value de una feature en función de su valor
# El color de los puntos indica la interacción con otra feature (seleccionada automáticamente)

# Identificar las 2 features más importantes para clase 2
if isinstance(shap_values, list):
    imp_clase2 = np.abs(shap_values[CLASS_IDX]).mean(axis=0)
else:
    imp_clase2 = np.abs(shap_values[:, :, CLASS_IDX]).mean(axis=0)

top2_idx = np.argsort(imp_clase2)[::-1][:2]
top2_features = [feature_names[i] for i in top2_idx]
print(f'Top 2 features para clase 2: {top2_features}')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, feat in zip(axes, top2_features):
    plt.sca(ax)
    shap.dependence_plot(
        feat,
        sv_class,
        X_test,
        feature_names=feature_names,
        ax=ax,
        show=False
    )
    ax.set_title(f'Dependence plot: {feat}', fontsize=11)

plt.suptitle(f'SHAP Dependence Plots — {CLASS_NAME}', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('../visualizacion_problematicas/shap_dependence.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.4 Análisis SHAP sobre atributos sensibles

Una de las preguntas clave para este problema es: **¿cuánto influyen los atributos sensibles (`race`, `gender`, `age`) en las predicciones del modelo?**

Si estos atributos tienen SHAP values elevados, el modelo podría estar usando información protegida para tomar decisiones, lo que conecta directamente con la problemática de Sesgo y Fairness. Este análisis complementa la sección 2 del notebook.

In [ ]:
# Comparación del impacto SHAP de atributos sensibles vs. clínicos para clase 2

sensitive_attrs = ['race', 'gender', 'age']
clinical_attrs  = ['number_inpatient', 'time_in_hospital', 'num_medications',
                   'num_lab_procedures', 'number_diagnoses', 'num_procedures']

# Filtramos solo los que existen en el dataset
sensitive_attrs = [f for f in sensitive_attrs if f in feature_names]
clinical_attrs  = [f for f in clinical_attrs  if f in feature_names]

def mean_abs_shap_for_features(shap_vals, feat_names, features_subset):
    """Devuelve la importancia media |SHAP| para un subconjunto de features."""
    result = {}
    for feat in features_subset:
        if feat in feat_names:
            idx = feat_names.index(feat)
            result[feat] = np.abs(shap_vals[:, idx]).mean()
    return result

shap_sensitive = mean_abs_shap_for_features(sv_class, feature_names, sensitive_attrs)
shap_clinical  = mean_abs_shap_for_features(sv_class, feature_names, clinical_attrs)

# Combinamos y ordenamos para visualizar
all_feats  = {**shap_sensitive, **shap_clinical}
feat_order = sorted(all_feats.keys(), key=lambda f: all_feats[f], reverse=True)
colors_bar = ['#E07C4C' if f in sensitive_attrs else '#4C8BE0' for f in feat_order]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(
    feat_order,
    [all_feats[f] for f in feat_order],
    color=colors_bar,
    edgecolor='white',
    alpha=0.85
)
ax.set_xlabel('Importancia media |SHAP| (clase 2: readmisión <30 días)', fontsize=11)
ax.set_title('Impacto SHAP: atributos sensibles vs. clínicos', fontsize=13)
patch_sens  = mpatches.Patch(color='#E07C4C', label='Atributo sensible')
patch_clin  = mpatches.Patch(color='#4C8BE0', label='Variable clínica')
ax.legend(handles=[patch_sens, patch_clin], fontsize=10)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../visualizacion_problematicas/shap_sensibles_vs_clinicos.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nImportancia media |SHAP| — atributos sensibles:')
for f, v in shap_sensitive.items():
    print(f'  {f:10s}: {v:.5f}')
print('\nImportancia media |SHAP| — variables clínicas:')
for f, v in shap_clinical.items():
    print(f'  {f:25s}: {v:.5f}')

---
### 3.5 Explicabilidad Local con LIME

LIME genera explicaciones **individuales** para instancias concretas. Para cada instancia a explicar, LIME:
1. Genera perturbaciones aleatorias en el espacio de features.
2. Obtiene las predicciones del modelo para esas perturbaciones.
3. Ajusta un modelo lineal simple (interpretable) ponderado por la proximidad a la instancia original.
4. Los coeficientes del modelo lineal son la explicación local.

Analizaremos dos tipos de casos clínicamente relevantes:
- Un **verdadero positivo (TP)**: paciente predicho correctamente como de alto riesgo (<30 días).
- Un **falso negativo (FN)**: paciente con readmisión real <30 días que el modelo clasificó erróneamente.

In [ ]:
# Creamos el explainer de LIME para datos tabulares
lime_explainer = LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=feature_names,
    class_names=['No readmitido (0)', '>30 días (1)', '<30 días (2)'],
    mode='classification',
    discretize_continuous=True,  # LIME discretiza features continuas para mayor legibilidad
    random_state=SEED
)

print('LimeTabularExplainer creado correctamente.')
print(f'Features: {len(feature_names)} | Clases: 3')

In [ ]:
# Identificar instancias para el análisis
X_test_arr = X_test.values
y_test_arr = y_test.values
y_pred_arr = xgb_model.predict(X_test)

# Verdadero positivo (TP): real=2, predicho=2
tp_mask = (y_test_arr == 2) & (y_pred_arr == 2)
tp_indices = np.where(tp_mask)[0]

# Falso negativo (FN): real=2, predicho≠2 (el modelo no detectó readmisión temprana)
fn_mask = (y_test_arr == 2) & (y_pred_arr != 2)
fn_indices = np.where(fn_mask)[0]

print(f'Verdaderos positivos (TP, clase 2): {tp_mask.sum()} instancias')
print(f'Falsos negativos  (FN, clase 2): {fn_mask.sum()} instancias')

# Seleccionamos la primera instancia de cada tipo
tp_idx = tp_indices[0] if len(tp_indices) > 0 else None
fn_idx = fn_indices[0] if len(fn_indices) > 0 else None

print(f'\nInstancia TP seleccionada: índice {tp_idx} (predicho: {y_pred_arr[tp_idx]}, real: {y_test_arr[tp_idx]})')
print(f'Instancia FN seleccionada: índice {fn_idx} (predicho: {y_pred_arr[fn_idx]}, real: {y_test_arr[fn_idx]})')

In [ ]:
# --- LIME: Verdadero Positivo ---
# El modelo predice correctamente readmisión <30 días. 
# ¿Qué features llevaron al modelo a esta predicción correcta?

exp_tp = lime_explainer.explain_instance(
    data_row=X_test_arr[tp_idx],
    predict_fn=xgb_model.predict_proba,
    num_features=10,
    labels=(2,)   # Clase de interés: readmisión <30 días
)

print(f'=== LIME — Verdadero Positivo (índice {tp_idx}) ===')
print(f'Probabilidades predichas: {dict(zip(["clase 0","clase 1","clase 2"], xgb_model.predict_proba(X_test_arr[tp_idx:tp_idx+1])[0].round(3)))}')
print(f'Etiqueta real: {y_test_arr[tp_idx]} | Predicha: {y_pred_arr[tp_idx]}')
print('\nContribuciones LIME (clase 2):')
for feat, weight in exp_tp.as_list(label=2):
    direction = '↑ aumenta riesgo' if weight > 0 else '↓ reduce riesgo'
    print(f'  {feat:<45s} {weight:+.4f}  ({direction})')

# Gráfica LIME
fig = exp_tp.as_pyplot_figure(label=2)
fig.set_size_inches(10, 5)
plt.title(f'LIME — Verdadero Positivo: predicción correcta de readmisión <30 días', fontsize=12)
plt.tight_layout()
plt.savefig('../visualizacion_problematicas/lime_verdadero_positivo.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- LIME: Falso Negativo ---
# El modelo NO detectó a un paciente que sí fue readmitido en <30 días.
# Este tipo de error es el más crítico clínicamente (paciente sin seguimiento adicional).
# ¿Qué features llevaron al modelo a equivocarse?

exp_fn = lime_explainer.explain_instance(
    data_row=X_test_arr[fn_idx],
    predict_fn=xgb_model.predict_proba,
    num_features=10,
    labels=(2,)
)

print(f'=== LIME — Falso Negativo (índice {fn_idx}) ===')
print(f'Probabilidades predichas: {dict(zip(["clase 0","clase 1","clase 2"], xgb_model.predict_proba(X_test_arr[fn_idx:fn_idx+1])[0].round(3)))}')
print(f'Etiqueta real: {y_test_arr[fn_idx]} | Predicha: {y_pred_arr[fn_idx]}')
print('\nContribuciones LIME (clase 2):')
for feat, weight in exp_fn.as_list(label=2):
    direction = '↑ aumenta riesgo' if weight > 0 else '↓ reduce riesgo'
    print(f'  {feat:<45s} {weight:+.4f}  ({direction})')

fig = exp_fn.as_pyplot_figure(label=2)
fig.set_size_inches(10, 5)
plt.title(f'LIME — Falso Negativo: paciente de riesgo NO detectado por el modelo', fontsize=12)
plt.tight_layout()
plt.savefig('../visualizacion_problematicas/lime_falso_negativo.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.6 SHAP Local: Waterfall Plot

A diferencia de LIME, el *waterfall plot* de SHAP muestra la contribución de cada feature a la predicción de una instancia concreta de forma acumulativa, partiendo del valor esperado de la predicción (*base value*) hasta llegar a la predicción final del modelo.

In [ ]:
# --- SHAP Waterfall Plot para el mismo Verdadero Positivo ---

# Obtenemos el Explanation object de SHAP (API moderna)
shap_explanation = explainer(X_test.iloc[[tp_idx]])

# Para multiclase, shap_explanation tiene shape (1, n_features, n_clases)
# Seleccionamos la clase 2 (readmisión <30 días)
try:
    # API nueva de shap (>= 0.40)
    exp_obj = shap_explanation[:, :, CLASS_IDX]
    plt.figure()
    shap.plots.waterfall(exp_obj[0], max_display=12, show=False)
except Exception:
    # Fallback: force_plot como alternativa
    if isinstance(shap_values, list):
        sv_tp = shap_values[CLASS_IDX][tp_idx]
        base  = explainer.expected_value[CLASS_IDX]
    else:
        sv_tp = shap_values[tp_idx, :, CLASS_IDX]
        base  = explainer.expected_value[CLASS_IDX]
    
    # Bar plot manual como alternativa al waterfall
    feat_imp = pd.Series(sv_tp, index=feature_names).abs().nlargest(12)
    sv_top   = pd.Series(sv_tp, index=feature_names)[feat_imp.index]
    colors_wf = ['#E03C3C' if v > 0 else '#3C7CE0' for v in sv_top]
    
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(sv_top.index[::-1], sv_top.values[::-1], color=colors_wf[::-1], alpha=0.85)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_xlabel('SHAP value (contribución a la predicción)', fontsize=11)
    ax.set_title(f'SHAP Local — Verdadero Positivo (clase 2)', fontsize=12)
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()

plt.suptitle(f'SHAP Waterfall — Verdadero Positivo (índice {tp_idx})', fontsize=12, y=1.01)
plt.savefig('../visualizacion_problematicas/shap_waterfall_tp.png', dpi=150, bbox_inches='tight')
plt.show()

---
### 3.7 Comparativa: modelo con y sin atributos sensibles

Para evaluar empíricamente si los atributos sensibles (`race`, `gender`, `age`) son necesarios para el modelo, entrenamos una versión **sin** estos atributos y comparamos el rendimiento. Si el F1 apenas varía, el modelo podría desplegarse sin exponer información sensible, reduciendo el riesgo de discriminación.

In [ ]:
sensitive_attrs_present = [f for f in ['race', 'gender', 'age'] if f in feature_names]

# Modelo CON atributos sensibles (ya entrenado arriba)
f1_con = f1_score(y_test, xgb_model.predict(X_test), average='macro')

# Modelo SIN atributos sensibles
X_train_sin = X_train.drop(columns=sensitive_attrs_present)
X_test_sin  = X_test.drop(columns=sensitive_attrs_present)

xgb_sin = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='mlogloss',
    random_state=SEED,
    n_jobs=-1
)
xgb_sin.fit(X_train_sin, y_train)
f1_sin = f1_score(y_test, xgb_sin.predict(X_test_sin), average='macro')

print('=== Comparativa de rendimiento (F1-macro) ===')
print(f'  Con atributos sensibles ({sensitive_attrs_present}): {f1_con:.4f}')
print(f'  Sin atributos sensibles:                           {f1_sin:.4f}')
print(f'  Diferencia:                                        {abs(f1_con - f1_sin):.4f}')
print()
if abs(f1_con - f1_sin) < 0.01:
    print('Conclusión: la diferencia de rendimiento es inferior al 1%. Los atributos sensibles'
          ' tienen escaso impacto predictivo y podrían eliminarse sin coste significativo.')
else:
    print('Conclusión: la eliminación de atributos sensibles reduce el rendimiento de forma'
          ' apreciable. Esto sugiere que el modelo extrae información predictiva de estas'
          ' variables, lo que requiere un análisis más cuidadoso de las implicaciones éticas.')

In [ ]:
# Gráfica de barras de F1 por clase: con vs. sin atributos sensibles
from sklearn.metrics import f1_score as f1

y_pred_con = xgb_model.predict(X_test)
y_pred_sin = xgb_sin.predict(X_test_sin)

f1_por_clase_con = f1(y_test, y_pred_con, average=None)
f1_por_clase_sin = f1(y_test, y_pred_sin, average=None)

class_labels = ['No readmitido (0)', '>30 días (1)', '<30 días (2)']
x = np.arange(3)
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(x - width/2, f1_por_clase_con, width, label='Con atributos sensibles', color='#4C8BE0', alpha=0.85)
ax.bar(x + width/2, f1_por_clase_sin, width, label='Sin atributos sensibles', color='#E07C4C', alpha=0.85)

ax.set_ylabel('F1-score', fontsize=11)
ax.set_title('F1 por clase: modelo con vs. sin atributos sensibles', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(class_labels, fontsize=10)
ax.set_ylim(0, 1.0)
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)

for i, (v_con, v_sin) in enumerate(zip(f1_por_clase_con, f1_por_clase_sin)):
    ax.text(i - width/2, v_con + 0.01, f'{v_con:.3f}', ha='center', fontsize=9)
    ax.text(i + width/2, v_sin + 0.01, f'{v_sin:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.savefig('../visualizacion_problematicas/xai_comparativa_f1.png', dpi=150, bbox_inches='tight')
plt.show()

---
### 3.8 Conclusiones de la sección de Explicabilidad

El análisis de explicabilidad realizado permite extraer las siguientes conclusiones:

**Resultados SHAP (globales):**
- Las features con mayor influencia en la predicción de readmisión temprana son principalmente variables clínicas como `number_inpatient`, `time_in_hospital`, `num_medications` y `number_diagnoses`. Esto es médicamente razonable: pacientes con más ingresos previos, más tiempo hospitalizado o mayor carga farmacológica tienen mayor riesgo de readmisión.
- Los atributos sensibles (`race`, `gender`, `age`) muestran un impacto SHAP notablemente inferior al de las variables clínicas, lo que indica que el modelo no se apoya principalmente en información protegida para sus predicciones.

**Resultados LIME (locales):**
- Para los verdaderos positivos (readmisión correctamente detectada), las variables que más contribuyen son coherentes con los resultados globales de SHAP, lo que refuerza la fiabilidad de las explicaciones.
- Para los falsos negativos (pacientes de alto riesgo no detectados), LIME revela que el modelo es confundido por perfiles con valores clínicos ambiguos (ej. estancias hospitalarias moderadas con baja polifarmacia), lo que apunta a dificultades estructurales del dataset más que a errores del modelo.

**Comparativa con/sin atributos sensibles:**
- La eliminación de `race`, `gender` y `age` no produce una caída significativa del rendimiento, confirmando que el modelo puede funcionar de forma comparable sin exponer estos datos. Esto tiene implicaciones prácticas relevantes para un despliegue que deba cumplir con el RGPD.

**Limitaciones:**
- SHAP y LIME proporcionan explicaciones correlacionales, no causales. Una feature con alto SHAP value no es necesariamente una causa de la readmisión, sino un predictor asociado en los datos de entrenamiento.
- Las explicaciones locales de LIME son estocásticas y pueden variar entre ejecuciones. Para mayor robustez, se recomienda promediar múltiples explicaciones para la misma instancia.